# Continuing the rejection-sampler posterior with MCMC

The rejection sampler in `harv` is designed to handle and map out the structure of complex, multi-modal RV posteriors. It can effectively survey the full prior volume and return a set of posterior samples even when the period is wildly multi-modal without having to consider convergence diagnostics or other metrics for assessing the quality of returned MCMC samples. But once we have isolated the mode(s) of interest (i.e., if the samples are unimodal in period), we usually want a dense posterior sampling with many independent samples for downstream uncertainty propagation. That's where MCMC can help.

`harv` provides an interface to run standard MCMC using `numpyro` via the {py:class}`~harv.NumpyroSampler`. This class builds a numpyro model from the same prior, parameterization, and extensions you used for rejection sampling, and can initialize from a `Samples` object outputted by a previous {py:class}`~harv.RejectionSampler` run.

In this tutorial we'll run rejection sampling on an APOGEE source with 28 visits and a clear orbit, including a {py:class}`~harv.Jitter` extension to absorb any underestimated per-visit error bars, and use the returned sample(s) to start 4 MCMC chains of NUTS sampling to generate a denser sampling of the posterior.

We assume familiarity with the basic rejection-sampling workflow from the {doc}`getting started tutorial <0-getting-started>`.

```{note}
Later, we will run 4 MCMC chains. If you are running this locally, you can often run these in parallel. This requires telling JAX up front how many host devices to expose, which we do via `numpyro.set_host_device_count(...)` before importing JAX. Uncomment this code to run locally. 
```

In [ ]:
# import numpyro

# numpyro.set_host_device_count(4)

In [ ]:
import astropy.table as at
import jax
import matplotlib.pyplot as plt
import numpyro.distributions as dist
import quaxed.numpy as jnp
from unxt import Q

import harv

jax.config.update("jax_enable_x64", True)

%matplotlib inline

## Loading the data

We use APOGEE DR17 visit RVs for source 2M03385429+4623449, which has 28 epochs and a clear, well-sampled orbital signal. This is exactly the regime where MCMC is worth the cost over rejection sampling alone.

In [ ]:
tbl = at.Table.read("../data/apogeedr17-2M03385429+4623449.csv")

fdtype = jnp.float64
data = harv.RVData(
    time=Q(tbl["JD"].astype(fdtype), "day"),
    rv=Q(tbl["VHELIO"].astype(fdtype), "km/s"),
    rv_err=Q(tbl["VRELERR"].astype(fdtype), "km/s"),
)
data

In [ ]:
_ = data.plot(relative_to_t_ref=True)

## Rejection sampling with a jitter extension

We set up the standard log-uniform period prior and etc. via {py:meth}`~harv.StandardRV.default_prior`, and add a {py:class}`~harv.Jitter` extension to allow extra (white) variance on top of the formal APOGEE error bars. The jitter parameter is sampled from a `HalfNormal(0.5 km/s)` prior. This is wide enough to allow up to ~few km/s of excess scatter if the data demand it, but with most prior mass at small jitter so we don't over-inflate the errors when they're already correct.

In [ ]:
prior = harv.StandardRV().default_prior(
    period_min=Q(100.0, "day"),
    period_max=Q(2000.0, "day"),
    sigma_K0=Q(30.0, "km/s"),
    sigma_v0=Q(50.0, "km/s"),
    jitter=harv.QD(dist.HalfNormal(0.5), "km/s"),
)

model = harv.RVModel(extensions=(harv.Jitter(param_unit="km/s"),))

In [ ]:
rej_sampler = harv.RejectionSampler(prior, model)
rej_samples = rej_sampler.run(
    data,
    n_prior_samples=10_000_000,
    max_posterior_samples=1024,
    seed=42,
)
len(rej_samples)

With 28 well-sampled epochs the period posterior should already be unimodal. And in fact, the rejection sampler returns only one sample because we used a very wide period prior. For your own use cases, the number of returned samples from the rejection sampler can depend strongly on the period prior. For example, if you use a restricted prior, you could end up with more samples for well-constrained cases like this one, but at the expense of missing other modes / orbital solutions in more complex cases (e.g., with fewer data points). 

Let's visualize the returned sample(s) from the rejection sampler:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax = harv.plot.plot_rv(
    rej_samples,
    data=data,
    extensions=rej_sampler.get_extensions(),
    n_samples=128,
    relative_to_t_ref=True,
    relative_to_median_v_sys=True,
    ax=ax,
)

Note that when jitter is included as an extension, the plotted data points are inflated (and drawn as red error bars by default) to include the median jitter value in addition to the formal error bars. In any case, for this data set, the returned sample from the rejection sampler looks like a good fit to the data.

## Continuing with MCMC

Now we want a dense posterior sampling, so we will use the sample(s) from the rejection sampler to initialize the MCMC chains. {py:class}`~harv.NumpyroSampler` takes the same prior and extensions we just used, and its `run()` method accepts an `init_samples` argument: rejection samples used to initialize each MCMC chain. With `num_chains=4`, the sampler picks 4 distinct rejection samples (cycling if there are fewer than 4) and uses them as the starting positions for 4 independent chains.

By default the sampler uses NUTS (no-U-turn HMC) with `marginalized=True`, meaning the linear parameters (`rv_semiamp`, `v_sys`) are analytically marginalized inside the likelihood and then conditionally sampled at the end so the returned `Samples` object still contains them. Only the nonlinear parameters (and any nonlinear extensions like jitter) are explored by HMC.

In [ ]:
mcmc_sampler = harv.NumpyroSampler(prior, model)

In [ ]:
mcmc_samples = mcmc_sampler.run(
    data,
    init_samples=rej_samples,
    num_chains=4,
    num_warmup=1000,
    num_samples=2000,
    seed=42,
)
mcmc_samples

The returned `Samples` flattens the chain dimension by default — with `num_chains=4` and `num_samples=2000` we get $4 \times 2000 = 8000$ samples. The chain count is preserved in the metadata in case we want to do per-chain diagnostics:

In [ ]:
print("n_samples =", mcmc_samples.n_samples)
print("num_chains =", mcmc_samples.metadata["num_chains"])

Numpyro internally returns an `arviz`-friendly xarray `DataTree` object. To get back the raw `DataTree` object, we can use the `to_arviz()` method on the returned `Samples` object:

In [ ]:
dt = mcmc_samples.to_arviz()
dt

We could use this object to assess the convergence of the MCMC sampling:

In [ ]:
import arviz as az

az.summary(dt)

## Step 3 — Comparing rejection and MCMC posteriors

Let's overlay the rejection and MCMC posteriors. The MCMC chains should fill in a smooth, well-sampled posterior in the neighborhood of the rejection mode:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), layout="constrained")

for ax, (xkey, ykey) in zip(
    axes,
    [("period", "eccentricity"), ("period", "rv_semiamp"), ("phase_peri", "arg_peri")],
    strict=True,
):
    rej_w = rej_samples.wrap_angles()
    mc_w = mcmc_samples.wrap_angles()
    ax.plot(rej_w[xkey].value, rej_w[ykey].value, ".", alpha=0.4, label="rejection")
    ax.plot(mc_w[xkey].value, mc_w[ykey].value, ".", alpha=0.1, label="MCMC")
    ax.set(xlabel=xkey, ylabel=ykey)
axes[0].legend(markerscale=2)

The jitter parameter is sampled by HMC alongside the orbital nonlinear parameters. Its marginal posterior tells us whether the formal APOGEE errors are sufficient or whether the data demand additional white-noise variance:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(mcmc_samples["jitter"].to_value("km/s"), bins=50, density=True)
ax.set(xlabel="jitter [km/s]", ylabel="posterior density")

{py:meth}`Samples.summary <harv.Samples.summary>` returns median + 16th/84th percentile statistics for any subset of parameters — handy for reporting:

In [ ]:
summ = mcmc_samples.wrap_angles().summary(
    ["period", "eccentricity", "arg_peri", "rv_semiamp", "v_sys", "jitter"]
)
print("Median values:")
for k, v in summ.items():
    print(f"{k}: {v['median']}")

Finally, we will plot posterior orbit curves over the data using the MCMC samples:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
_ = harv.plot.plot_rv(
    mcmc_samples,
    data=data,
    extensions=model.extensions,
    n_samples=512,
    ax=axes[0],
)
_ = harv.plot.plot_rv(
    mcmc_samples,
    data=data,
    extensions=model.extensions,
    phase_fold_median=True,
    ax=axes[1],
)
axes[0].set(title="RV curve")
axes[1].set(title="Phase-folded RV curve")